In [1]:
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql.window import Window
from pyspark.sql.functions import col, sum

In [3]:
spark = (
    SparkSession.builder
    .appName("Customer_Churn_Pipeline")
    .enableHiveSupport()
    .getOrCreate()
)

# Create Hive Database

In [5]:
HIVE_DB = "customer_churn_db"

spark.sql(f"""
    CREATE DATABASE IF NOT EXISTS {HIVE_DB}
""")

In [6]:
spark.sql(f"USE {HIVE_DB}")

print("Available databases:")
spark.sql("SHOW DATABASES").show()

print("Current database:")
spark.sql("SELECT current_database()").show()

Available databases:
+-----------------+
|        namespace|
+-----------------+
|customer_churn_db|
|          default|
+-----------------+

Current database:
+------------------+
|current_database()|
+------------------+
| customer_churn_db|
+------------------+



## CONFIGURATION

In [7]:
BASE_PATH = "/user/student/Capstone_project"

# Input Paths
CUSTOMER_PATH = f"{BASE_PATH}/raw/customers"
TICKETS_PATH = f"{BASE_PATH}/raw/customer_support_tickets"
OFFERS_PATH = f"{BASE_PATH}/raw/offers"
CUSTOMER_USAGE_PATH = f"{BASE_PATH}/raw/usages"

# Output Paths
SILVER_PATH = f"{BASE_PATH}/silver"
GOLD_PATH = f"{BASE_PATH}/gold"

# Output Table Paths
CUSTOMER_SILVER_PATH = f"{SILVER_PATH}/customers"
CUSTOMER_USAGE_SILVER_PATH = f"{SILVER_PATH}/customer_usages"
TICKETS_SILVER_PATH = f"{SILVER_PATH}/tickets"
OFFERS_SILVER_PATH = f"{SILVER_PATH}/offers"

DIM_CUSTOMER_PATH = f"{GOLD_PATH}/dim_customer"
DIM_DATE_PATH = f"{GOLD_PATH}/dim_date"
FACT_ACTIVITY_PATH = f"{GOLD_PATH}/fact_customer_monthly_activity"

# Gold Tables (datawarehouse tables)
DIM_CUSTOMER_TABLE = f"{HIVE_DB}.dim_customer"
DIM_DATE_TABLE = f"{HIVE_DB}.dim_date"
FACT_ACTIVITY_TABLE = f"{HIVE_DB}.fact_customer_monthly_activity"

# File Format
INPUT_FORMAT = "parquet"
OUTPUT_FORMAT = "parquet"

## READ DATA

In [8]:
customer_df = (
    spark.read
    .format(INPUT_FORMAT)
    .option("header", "true")
    .load(CUSTOMER_PATH)
)

customer_df.show(3, truncate=False)

+----------+-----------+--------+------------+---------+------+---+------+---------------+-----------+----------------+----------------+------+-------------+----------+----------------------------------+------------+--------------+----------------------+
|row_number|customer_id|surname |credit_score|geography|gender|age|tenure|num_of_products|has_cr_card|is_active_member|estimated_salary|exited|date_of_birth|join_date |email                             |phone_number|national_id   |address               |
+----------+-----------+--------+------------+---------+------+---+------+---------------+-----------+----------------+----------------+------+-------------+----------+----------------------------------+------------+--------------+----------------------+
|1         |15634602   |Hargrave|619         |France   |Female|42 |2     |1              |1          |1               |101348.88       |1     |1982-02-07   |2001-06-07|maria.hargrave602@examplebank.test|208730121132|30478297832259|163 

In [9]:
customer_usage_df = (
    spark.read
    .format(INPUT_FORMAT)
    .option("header", "true")
    .load(CUSTOMER_USAGE_PATH)
)

customer_usage_df.show(3, truncate=False)

+--------+-----------+------------+---------------+------------+----------+
|usage_id|customer_id|product_type|monthly_balance|num_products|date      |
+--------+-----------+------------+---------------+------------+----------+
|1       |15665177   |Checking    |445.51         |2           |2026/04/08|
|2       |15756472   |Savings     |null           |1           |2025/12/19|
|3       |15589881   |Savings     |320.82         |2           |2026/04/05|
+--------+-----------+------------+---------------+------------+----------+
only showing top 3 rows



In [10]:
tickets_df = (
    spark.read
    .format(INPUT_FORMAT)
    .option("header", "true")
    .load(TICKETS_PATH)
)

tickets_df.show(3, truncate=False)

+---------+-----------+---------------+--------+-------------------+----------+
|ticket_id|customer_id|issue_type     |severity|resolution_time_hrs|date      |
+---------+-----------+---------------+--------+-------------------+----------+
|1        |15624343   |Login Issue    |Medium  |15.1               |2026/07/27|
|2        |15633930   |Payment Failure|Critical|2.1                |2025/10/30|
|3        |15793854   |Complaint      |High    |3.5                |2025/09/15|
+---------+-----------+---------------+--------+-------------------+----------+
only showing top 3 rows



In [11]:
offers_df = (
    spark.read
    .format(INPUT_FORMAT)
    .option("header", "true")
    .load(OFFERS_PATH)
)

offers_df.show(3, truncate=False)

+--------+-----------+---------------+--------+------------+
|offer_id|customer_id|offer_type     |accepted|date_offered|
+--------+-----------+---------------+--------+------------+
|1       |15784148   |Cashback       |True    |2025/09/03  |
|2       |15687329   |Cashback       |True    |2025/09/04  |
|3       |15738150   |Investment Plan|True    |2025/09/05  |
+--------+-----------+---------------+--------+------------+
only showing top 3 rows



## EDA

In [12]:
print("Customer rows:", customer_df.count())
print("Customer Usage rows:", customer_usage_df.count())
print("Tickets rows:", tickets_df.count())
print("Offers rows:", offers_df.count())

Customer rows: 10000
Customer Usage rows: 15452
Tickets rows: 6059
Offers rows: 8079


In [13]:
print("Customer columns:", len(customer_df.columns))
print("Customer Usage columns:", len(customer_usage_df.columns))
print("Tickets columns:", len(tickets_df.columns))
print("Offers columns:", len(offers_df.columns))

Customer columns: 19
Customer Usage columns: 6
Tickets columns: 6
Offers columns: 5


In [14]:
tables = [
    ("Customer", customer_df),
    ("Customer Usage", customer_usage_df),
    ("Tickets", tickets_df),
    ("Offers", offers_df)
]

for table_name, table in tables:
    print(f"{table_name} Schema:")
    table.printSchema()
    print("*" * 50)

Customer Schema:
root
 |-- row_number: integer (nullable = true)
 |-- customer_id: integer (nullable = true)
 |-- surname: string (nullable = true)
 |-- credit_score: integer (nullable = true)
 |-- geography: string (nullable = true)
 |-- gender: string (nullable = true)
 |-- age: integer (nullable = true)
 |-- tenure: integer (nullable = true)
 |-- num_of_products: integer (nullable = true)
 |-- has_cr_card: integer (nullable = true)
 |-- is_active_member: integer (nullable = true)
 |-- estimated_salary: double (nullable = true)
 |-- exited: integer (nullable = true)
 |-- date_of_birth: string (nullable = true)
 |-- join_date: string (nullable = true)
 |-- email: string (nullable = true)
 |-- phone_number: long (nullable = true)
 |-- national_id: long (nullable = true)
 |-- address: string (nullable = true)

**************************************************
Customer Usage Schema:
root
 |-- usage_id: integer (nullable = true)
 |-- customer_id: integer (nullable = true)
 |-- product_ty

In [15]:
def null_analysis(df, table_name):
    total_rows = df.count()

    print(f"\n{'=' * 60}")
    print(f"{table_name} - NULL Analysis")
    print(f"Total Rows: {total_rows}")
    print(f"{'=' * 60}")

    result = df.select([
        sum(
            col(c).isNull().cast("int")
        ).alias(f"{c}_nulls")
        for c in df.columns
    ])

    result.show()

In [16]:
null_analysis(customer_df, "Customer")
null_analysis(customer_usage_df, "Customer Usage")
null_analysis(tickets_df, "Tickets")
null_analysis(offers_df, "Offers")


Customer - NULL Analysis
Total Rows: 10000
+----------------+-----------------+-------------+------------------+---------------+------------+---------+------------+---------------------+-----------------+----------------------+----------------------+------------+-------------------+---------------+-----------+------------------+-----------------+-------------+
|row_number_nulls|customer_id_nulls|surname_nulls|credit_score_nulls|geography_nulls|gender_nulls|age_nulls|tenure_nulls|num_of_products_nulls|has_cr_card_nulls|is_active_member_nulls|estimated_salary_nulls|exited_nulls|date_of_birth_nulls|join_date_nulls|email_nulls|phone_number_nulls|national_id_nulls|address_nulls|
+----------------+-----------------+-------------+------------------+---------------+------------+---------+------------+---------------------+-----------------+----------------------+----------------------+------------+-------------------+---------------+-----------+------------------+-----------------+-----------

In [17]:
tickets_df.filter(
    F.col("issue_type").isNull()
).show(
    20,
    truncate=False
)

+---------+-----------+----------+--------+-------------------+----------+
|ticket_id|customer_id|issue_type|severity|resolution_time_hrs|date      |
+---------+-----------+----------+--------+-------------------+----------+
|6        |15638257   |null      |Low     |61.3               |2025/12/26|
|42       |15685844   |null      |Medium  |27.0               |2025/10/20|
|50       |15681506   |null      |Medium  |25.1               |2026/04/26|
|81       |15748595   |null      |Medium  |22.1               |2026/01/17|
|145      |15794048   |null      |Medium  |20.1               |2026/07/05|
|168      |15629846   |null      |Low     |19.5               |2025/10/27|
|204      |15789097   |null      |High    |6.0                |2026/05/31|
|208      |15711455   |null      |Medium  |29.7               |2026/06/06|
|260      |15794849   |null      |Medium  |13.8               |2026/03/03|
|265      |15756538   |null      |Low     |126.1              |2026/08/13|
|330      |15771086   |nu

In [18]:
tickets_df.filter(
    F.col("resolution_time_hrs").isNull()
).show(
    20,
    truncate=False
)

+---------+-----------+---------------+--------+-------------------+----------+
|ticket_id|customer_id|issue_type     |severity|resolution_time_hrs|date      |
+---------+-----------+---------------+--------+-------------------+----------+
|5        |15815070   |Fraud Alert    |Medium  |null               |2026/01/06|
|17       |15635125   |App Bug        |Low     |null               |2025/12/23|
|25       |15807312   |Login Issue    |Low     |null               |2026/08/28|
|53       |15796071   |Card Problem   |High    |null               |2026/03/15|
|100      |15701524   |App Bug        |High    |null               |2026/02/16|
|122      |15782659   |Fraud Alert    |High    |null               |2026/06/05|
|131      |15698246   |Login Issue    |Medium  |null               |2026/05/21|
|187      |15580682   |Fraud Alert    |Critical|null               |2026/02/07|
|190      |15608541   |Payment Failure|Medium  |null               |2026/01/04|
|238      |15784280   |Fraud Alert    |L

In [19]:
tickets_df.groupBy(
    "resolution_time_hrs"
).count().orderBy(
    F.col("resolution_time_hrs").asc_nulls_first()
).show(20)

+-------------------+-----+
|resolution_time_hrs|count|
+-------------------+-----+
|               null|  297|
|               -5.0|   93|
|                0.5|   64|
|                0.6|   10|
|                0.7|   13|
|                0.8|   18|
|                0.9|   17|
|                1.0|   25|
|                1.1|   24|
|                1.2|   27|
|                1.3|   34|
|                1.4|   30|
|                1.5|   29|
|                1.6|   40|
|                1.7|   48|
|                1.8|   51|
|                1.9|   56|
|                2.0|   40|
|                2.1|   47|
|                2.2|   44|
+-------------------+-----+
only showing top 20 rows



## Customer EDA

In [20]:
customer_df.describe().show()

+-------+------------------+-----------------+-------+-----------------+---------+------+------------------+------------------+------------------+-------------------+-------------------+-----------------+-------------------+-------------+----------+--------------------+--------------------+--------------------+--------------------+
|summary|        row_number|      customer_id|surname|     credit_score|geography|gender|               age|            tenure|   num_of_products|        has_cr_card|   is_active_member| estimated_salary|             exited|date_of_birth| join_date|               email|        phone_number|         national_id|             address|
+-------+------------------+-----------------+-------+-----------------+---------+------+------------------+------------------+------------------+-------------------+-------------------+-----------------+-------------------+-------------+----------+--------------------+--------------------+--------------------+--------------------

In [21]:
customer_df.groupBy("geography").count().show()

customer_df.groupBy("gender").count().show()

customer_df.groupBy("has_cr_card").count().show()

customer_df.groupBy("is_active_member").count().show()

customer_df.groupBy("exited").count().show()

+---------+-----+
|geography|count|
+---------+-----+
|  Germany| 2509|
|   France| 5014|
|    Spain| 2477|
+---------+-----+

+------+-----+
|gender|count|
+------+-----+
|Female| 4543|
|  Male| 5457|
+------+-----+

+-----------+-----+
|has_cr_card|count|
+-----------+-----+
|          1| 7055|
|          0| 2945|
+-----------+-----+

+----------------+-----+
|is_active_member|count|
+----------------+-----+
|               1| 5151|
|               0| 4849|
+----------------+-----+

+------+-----+
|exited|count|
+------+-----+
|     1| 2037|
|     0| 7963|
+------+-----+



In [22]:
print("Total customers:", customer_df.count())

print(
    "Distinct customer IDs:",
    customer_df.select("customer_id").distinct().count()
)

print(
    "Duplicate customer IDs:",
    customer_df.count()
    - customer_df.select("customer_id").distinct().count()
)

Total customers: 10000


Distinct customer IDs: 10000


Duplicate customer IDs: 0


In [24]:
customer_df.select(
    F.min("age").alias("min_age"),
    F.max("age").alias("max_age"),
    F.avg("age").alias("avg_age")
).show()

+-------+-------+-------+
|min_age|max_age|avg_age|
+-------+-------+-------+
|     18|     92|38.9218|
+-------+-------+-------+



In [25]:
customer_df.filter(
    (F.col("age") < 18) |
    (F.col("age") > 100)
).show()

+----------+-----------+-------+------------+---------+------+---+------+---------------+-----------+----------------+----------------+------+-------------+---------+-----+------------+-----------+-------+
|row_number|customer_id|surname|credit_score|geography|gender|age|tenure|num_of_products|has_cr_card|is_active_member|estimated_salary|exited|date_of_birth|join_date|email|phone_number|national_id|address|
+----------+-----------+-------+------------+---------+------+---+------+---------------+-----------+----------------+----------------+------+-------------+---------+-----+------------+-----------+-------+
+----------+-----------+-------+------------+---------+------+---+------+---------------+-----------+----------------+----------------+------+-------------+---------+-----+------------+-----------+-------+



## Customer Usage Analysis

In [26]:
customer_usage_df.groupBy("product_type").count().show()

+------------+-----+
|product_type|count|
+------------+-----+
|        Loan| 2945|
|        loan|   34|
|       Loan |   37|
| Credit Card| 2812|
|     Savings| 2884|
|  investment|   41|
|        null|  470|
|    checking|   40|
|    CHECKING|   34|
|Credit Card |   30|
|        LOAN|   49|
|     SAVINGS|   33|
|   Checking |   49|
|  INVESTMENT|   33|
|     savings|   31|
| CREDIT CARD|   47|
| credit card|   38|
|  Investment| 2863|
| Investment |   28|
|    Savings |   39|
+------------+-----+
only showing top 20 rows



In [27]:
customer_usage_df.select(
    F.min("monthly_balance").alias("min_balance"),
    F.max("monthly_balance").alias("max_balance"),
    F.avg("monthly_balance").alias("avg_balance")
).show()

+-----------+-------------+-----------------+
|min_balance|  max_balance|      avg_balance|
+-----------+-------------+-----------------+
|   -4991.08|4.995428372E7|366907.1775036686|
+-----------+-------------+-----------------+



In [28]:
# Check negative balances
customer_usage_df.filter(
    F.col("monthly_balance") < 0
).select(
    "usage_id",
    "customer_id",
    "monthly_balance"
).show(20)

+--------+-----------+---------------+
|usage_id|customer_id|monthly_balance|
+--------+-----------+---------------+
|     206|   15803842|         -661.9|
|     209|   15642885|       -3706.16|
|     239|   15628303|       -4932.44|
|     447|   15716236|       -4833.28|
|     502|   15770968|       -4074.18|
|     625|   15587584|       -3232.98|
|     741|   15715715|       -2439.78|
|     867|   15756560|       -3276.63|
|    1179|   15729956|       -3318.69|
|    1188|   15729454|        -2089.0|
|    1212|   15616172|        -484.46|
|    1238|   15678057|       -3416.85|
|    1297|   15621064|        -1474.0|
|    1349|   15718531|       -1099.39|
|    1362|   15622192|        -512.96|
|    1424|   15622532|       -2343.37|
|    1532|   15658969|       -1520.55|
|    1595|   15783276|        -4255.7|
|    1648|   15569917|       -3790.53|
|    1664|   15654878|       -3013.18|
+--------+-----------+---------------+
only showing top 20 rows



In [29]:
print(
    "Negative balances:",
    customer_usage_df
    .filter(F.col("monthly_balance") < 0)
    .count()
)

Negative balances: 228


In [30]:
customer_usage_df.select(
    "usage_id",
    "customer_id",
    "monthly_balance"
).orderBy(
    F.desc("monthly_balance")
).show(20)

+--------+-----------+---------------+
|usage_id|customer_id|monthly_balance|
+--------+-----------+---------------+
|   12084|   15750014|  4.995428372E7|
|    3634|   15656121|  4.963534048E7|
|   14919|   15734948|  4.927400661E7|
|    1337|   15580912|  4.892687494E7|
|     587|   15576022|  4.892229387E7|
|   14265|   15743075|  4.867158456E7|
|   13549|   15797381|  4.858786663E7|
|   14679|   15703399|  4.857490291E7|
|    1489|   15792004|  4.838675049E7|
|    1356|   15813444|  4.813151348E7|
|   15258|   15769948|   4.80278213E7|
|    7997|   15676526|  4.758513047E7|
|   10150|   15794356|  4.673735736E7|
|    5162|   15721730|   4.66732826E7|
|   11002|   15771997|  4.619881965E7|
|    2802|   15759066|  4.593122433E7|
|    7869|   15569364|  4.586389774E7|
|   13102|   15599182|  4.571929657E7|
|   12136|   15754494|  4.571596042E7|
|    3413|   15725039|  4.545885056E7|
+--------+-----------+---------------+
only showing top 20 rows



In [31]:
customer_usage_df.select(
    F.expr("percentile_approx(monthly_balance, array(0.01, 0.05, 0.25, 0.5, 0.75, 0.95, 0.99))")
    .alias("percentiles")
).show(truncate=False)

+----------------------------------------------------------------------+
|percentiles                                                           |
+----------------------------------------------------------------------+
|[-1859.89, 84.39, 569.99, 37740.3, 84731.19, 170916.96, 1.133334517E7]|
+----------------------------------------------------------------------+



In [32]:
customer_usage_df.filter(
    F.col("monthly_balance") < 0
).groupBy(
    "product_type"
).agg(
    F.count("*").alias("negative_records"),
    F.min("monthly_balance").alias("min_balance"),
    F.avg("monthly_balance").alias("avg_balance")
).orderBy(
    F.desc("negative_records")
).show()

+------------+----------------+-----------+-------------------+
|product_type|negative_records|min_balance|        avg_balance|
+------------+----------------+-----------+-------------------+
|    Checking|              50|   -4932.44|-2918.9067999999997|
| Credit Card|              43|   -4991.08| -2588.203255813953|
|        Loan|              42|   -4957.96|-2766.4130952380956|
|     Savings|              40|   -4892.98|-2443.0840000000003|
|  Investment|              37|   -4725.61|-2437.4037837837836|
|        null|              10|   -4769.57|-2526.1150000000002|
|    CHECKING|               2|   -3462.24|-3347.6099999999997|
|       Loan |               1|    -352.65|            -352.65|
|  investment|               1|    -139.59|            -139.59|
|    checking|               1|    -935.64|            -935.64|
| Investment |               1|   -2619.37|           -2619.37|
+------------+----------------+-----------+-------------------+



In [33]:
customer_usage_df.orderBy(
    F.desc("monthly_balance")
).select(
    "usage_id",
    "customer_id",
    "product_type",
    "monthly_balance",
    "num_products"
).show(20)

+--------+-----------+------------+---------------+------------+
|usage_id|customer_id|product_type|monthly_balance|num_products|
+--------+-----------+------------+---------------+------------+
|   12084|   15750014|     Savings|  4.995428372E7|           2|
|    3634|   15656121|    Checking|  4.963534048E7|           2|
|   14919|   15734948|        Loan|  4.927400661E7|           2|
|    1337|   15580912|     Savings|  4.892687494E7|           2|
|     587|   15576022|  Investment|  4.892229387E7|           2|
|   14265|   15743075|    Checking|  4.867158456E7|           2|
|   13549|   15797381|  Investment|  4.858786663E7|           2|
|   14679|   15703399|     Savings|  4.857490291E7|           2|
|    1489|   15792004| Credit Card|  4.838675049E7|           2|
|    1356|   15813444|  Investment|  4.813151348E7|           2|
|   15258|   15769948|        Loan|   4.80278213E7|           1|
|    7997|   15676526|        null|  4.758513047E7|           1|
|   10150|   15794356|   

## Tickets Analysis

In [34]:
tickets_df.groupBy("issue_type").count().orderBy(
    F.desc("count")
).show(50, truncate=False)

+---------------+-----+
|issue_type     |count|
+---------------+-----+
|Loan Query     |780  |
|Payment Failure|755  |
|Account Update |754  |
|App Bug        |723  |
|Complaint      |718  |
|Card Problem   |702  |
|Fraud Alert    |690  |
|Login Issue    |674  |
|null           |175  |
|loginn issue   |88   |
+---------------+-----+



In [35]:
tickets_df.groupBy("severity").count().show()

+----------+-----+
|  severity|count|
+----------+-----+
|  critical|   11|
|      High| 1138|
|       low|   26|
|       Low| 2036|
|     Low  |   28|
|      Low |   26|
|    Medium| 1738|
| Critical |   16|
|      HIGH|   15|
|       LOW|   27|
|      high|    8|
|    medium|   26|
|    High  |   13|
|    MEDIUM|   27|
|   Medium |   21|
|     High |   19|
|  CRITICAL|    6|
|  Medium  |   24|
|Critical  |    8|
|  Critical|  846|
+----------+-----+



In [36]:
F.initcap(
    F.trim(
        F.lower(F.col("severity"))
    )
)

Column<'initcap(trim(lower(severity)))'>

In [37]:
tickets_df.select(
    F.min("resolution_time_hrs").alias("min_resolution"),
    F.max("resolution_time_hrs").alias("max_resolution"),
    F.avg("resolution_time_hrs").alias("avg_resolution")
).show()

+--------------+--------------+------------------+
|min_resolution|max_resolution|    avg_resolution|
+--------------+--------------+------------------+
|          -5.0|       99999.0|1091.9059701492517|
+--------------+--------------+------------------+



In [38]:
tickets_df.filter(
    F.col("resolution_time_hrs") < 0
).show()

+---------+-----------+---------------+--------+-------------------+----------+
|ticket_id|customer_id|     issue_type|severity|resolution_time_hrs|      date|
+---------+-----------+---------------+--------+-------------------+----------+
|       23|   15803365| Account Update|    High|               -5.0|2025/12/13|
|      115|   15798615|   Card Problem|     Low|               -5.0|2026/07/29|
|      274|   15704053|    Fraud Alert|     Low|               -5.0|2025/12/10|
|      283|   15643359|        App Bug|     Low|               -5.0|2026/01/08|
|      366|   15626474|    Login Issue|     Low|               -5.0|2025/10/10|
|      419|   15624866|      Complaint|    High|               -5.0|2026/01/22|
|      504|   15614813|Payment Failure|  Medium|               -5.0|2026/01/16|
|      576|   15704681|      Complaint|    High|               -5.0|2026/08/06|
|      670|   15596575|    Fraud Alert|Critical|               -5.0|2026/06/07|
|      712|   15664802|     Loan Query| 

In [39]:
tickets_df.filter(
    F.col("resolution_time_hrs").isNull()
).count()

297

## Offers Analysis

In [40]:
offers_df.groupBy("offer_type").count().show()

+-------------------+-----+
|         offer_type|count|
+-------------------+-----+
|    Investment Plan| 1520|
|           Cashback| 1485|
|               null|  324|
|      Personal Loan| 1601|
|   Insurance Bundle| 1568|
|Credit Card Upgrade| 1581|
+-------------------+-----+



In [41]:
offers_df.groupBy("accepted").count().show()

+--------+-----+
|accepted|count|
+--------+-----+
|       0|  210|
|   False| 4618|
|   MAYBE|    1|
|       Y|   94|
|       N|  200|
|      No|  229|
|     Yes|  112|
|   false|  183|
|       1|   93|
|    True| 2245|
|    true|   94|
+--------+-----+



## Dates Analysis

In [42]:
customer_usage_df.select(
    "date"
).show(10)

+----------+
|      date|
+----------+
|2026/04/08|
|2025/12/19|
|2026/04/05|
|2026/08/27|
|2026/03/12|
|2026/03/08|
|2026/03/27|
|2026/01/24|
|2026/04/21|
|2025/12/04|
+----------+
only showing top 10 rows



In [43]:
tickets_df.select(
    "date"
).show(10)

+----------+
|      date|
+----------+
|2026/07/27|
|2025/10/30|
|2025/09/15|
|2026/01/21|
|2026/01/06|
|2025/12/26|
|2025/11/13|
|2025/10/25|
|2026/08/15|
|2026/06/09|
+----------+
only showing top 10 rows



In [44]:
offers_df.select(
    "date_offered"
).show(10)

+------------+
|date_offered|
+------------+
|  2025/09/03|
|  2025/09/04|
|  2025/09/05|
|  2025/09/06|
|  2025/09/07|
|  2025/09/08|
|  2025/09/09|
|  2025/09/10|
|  2025/09/11|
|  2025/09/12|
+------------+
only showing top 10 rows



In [45]:
customer_df.select(
    "date_of_birth",
    "join_date"
).show(10)

+-------------+----------+
|date_of_birth| join_date|
+-------------+----------+
|   1982-02-07|2001-06-07|
|   1983-12-19|2010-05-13|
|   1982-08-13|2003-06-12|
|   1985-09-08|2005-05-01|
|   1981-11-09|2007-07-11|
|   1980-03-27|1999-04-17|
|   1974-03-04|1996-12-26|
|   1995-12-15|2014-05-16|
|   1980-11-14|2001-05-27|
|   1997-09-03|2021-06-02|
+-------------+----------+
only showing top 10 rows



## CLEANING (Silver layer)

### Customer Cleaning: remove duplicate customers and convert date fields from Unix timestamps to proper Date types.

In [46]:
customer_clean_df = (
    customer_df
    .dropDuplicates(["customer_id"])
    .withColumn(
        "date_of_birth",
        F.to_date(F.from_unixtime(F.col("date_of_birth") / 1000))
    )
    .withColumn(
        "join_date",
        F.to_date(F.from_unixtime(F.col("join_date") / 1000))
    )
)

In [47]:
customer_clean_df.printSchema()

root
 |-- row_number: integer (nullable = true)
 |-- customer_id: integer (nullable = true)
 |-- surname: string (nullable = true)
 |-- credit_score: integer (nullable = true)
 |-- geography: string (nullable = true)
 |-- gender: string (nullable = true)
 |-- age: integer (nullable = true)
 |-- tenure: integer (nullable = true)
 |-- num_of_products: integer (nullable = true)
 |-- has_cr_card: integer (nullable = true)
 |-- is_active_member: integer (nullable = true)
 |-- estimated_salary: double (nullable = true)
 |-- exited: integer (nullable = true)
 |-- date_of_birth: date (nullable = true)
 |-- join_date: date (nullable = true)
 |-- email: string (nullable = true)
 |-- phone_number: long (nullable = true)
 |-- national_id: long (nullable = true)
 |-- address: string (nullable = true)



### Customer Usage Cleaning: standardize product types, handle missing values, convert dates, and impute monthly balance using the median per product type.

In [48]:
customer_usage_clean_df = (
    customer_usage_df
    .withColumn(
        "product_type",
        F.initcap(F.trim(F.lower(F.col("product_type"))))
    )
    .withColumn(
        "product_type",
        F.when(
            F.col("product_type") == "Creditcard",
            "Credit Card"
        ).otherwise(F.col("product_type"))
    )
    .withColumn(
        "product_type",
        F.coalesce(
            F.col("product_type"),
            F.lit("Unknown")
        )
    )
    .withColumn(
    "date",
    F.to_date(F.col("date"), "yyyy/MM/dd")
    )
    
)

In [49]:
usage_medians = (
    customer_usage_clean_df
    .groupBy("product_type")
    .agg(
        F.expr(
            "percentile_approx(monthly_balance, 0.5)"
        ).alias("median_balance")
    )
)

customer_usage_clean_df = (
    customer_usage_clean_df
    .join(
        usage_medians,
        on="product_type",
        how="left"
    )
    .withColumn(
        "monthly_balance",
        F.coalesce(
            F.col("monthly_balance"),
            F.col("median_balance")
        )
    )
    .drop("median_balance")
)

In [50]:
customer_usage_clean_df = (
    customer_usage_clean_df
    .filter(F.col("monthly_balance") >= 0)
)

In [51]:
print(
    "Negative balances:",
    customer_usage_clean_df
    .filter(F.col("monthly_balance") < 0)
    .count()
)

Negative balances: 0


In [52]:
customer_usage_clean_df.printSchema()

root
 |-- product_type: string (nullable = false)
 |-- usage_id: integer (nullable = true)
 |-- customer_id: integer (nullable = true)
 |-- monthly_balance: double (nullable = true)
 |-- num_products: integer (nullable = true)
 |-- date: date (nullable = true)



In [53]:
customer_usage_clean_df.select(
    "usage_id",
    "customer_id",
    "product_type",
    "monthly_balance",
    "date"
).show(10, truncate=False)

+--------+-----------+------------+---------------+----------+
|usage_id|customer_id|product_type|monthly_balance|date      |
+--------+-----------+------------+---------------+----------+
|1       |15665177   |Checking    |445.51         |2026-04-08|
|2       |15756472   |Savings     |36053.68       |2025-12-19|
|3       |15589881   |Savings     |320.82         |2026-04-05|
|4       |15584091   |Savings     |54886.26       |2026-08-27|
|5       |15653306   |Checking    |60361.31       |2026-03-12|
|6       |15762392   |Checking    |103960.52      |2026-03-08|
|7       |15725639   |Credit Card |130175.17      |2026-03-27|
|8       |15725002   |Checking    |603.99         |2026-01-24|
|9       |15568982   |Savings     |986.58         |2026-04-21|
|10      |15802466   |Credit Card |868.64         |2025-12-04|
+--------+-----------+------------+---------------+----------+
only showing top 10 rows



### Tickets Cleaning: standardize issue types and severity, handle missing and invalid resolution times, and convert ticket dates to Date type.

In [54]:
tickets_clean_df = (
    tickets_df
    .withColumn(
        "severity",
        F.initcap(F.trim(F.lower(F.col("severity"))))
    )
    .withColumn(
        "issue_type",
        F.when(
            F.lower(F.trim(F.col("issue_type"))) == "loginn issue",
            "Login Issue"
        )
        .otherwise(F.trim(F.col("issue_type")))
    )
    .withColumn(
        "issue_type",
        F.coalesce(
            F.col("issue_type"),
            F.lit("Unknown")
        )
    )
    .withColumn(
        "date",
        F.to_date(F.col("date"), "yyyy/MM/dd")
    )
    .withColumn(
        "resolution_time_hrs",
        F.coalesce(
            F.col("resolution_time_hrs"),
            F.lit(0)
        )
    )
    .withColumn(
        "is_resolved",
        F.when(
            F.col("resolution_time_hrs") > 0,
            1
        )
        .otherwise(0)
    )
)

In [55]:
tickets_clean_df.show(3)

+---------+-----------+---------------+--------+-------------------+----------+-----------+
|ticket_id|customer_id|     issue_type|severity|resolution_time_hrs|      date|is_resolved|
+---------+-----------+---------------+--------+-------------------+----------+-----------+
|        1|   15624343|    Login Issue|  Medium|               15.1|2026-07-27|          1|
|        2|   15633930|Payment Failure|Critical|                2.1|2025-10-30|          1|
|        3|   15793854|      Complaint|    High|                3.5|2025-09-15|          1|
+---------+-----------+---------------+--------+-------------------+----------+-----------+
only showing top 3 rows



In [56]:
tickets_clean_df.printSchema()

root
 |-- ticket_id: integer (nullable = true)
 |-- customer_id: integer (nullable = true)
 |-- issue_type: string (nullable = false)
 |-- severity: string (nullable = true)
 |-- resolution_time_hrs: double (nullable = false)
 |-- date: date (nullable = true)
 |-- is_resolved: integer (nullable = false)



### Offers Cleaning: standardize offer types, normalize acceptance values, handle invalid outcomes, and convert offer dates to Date type.

In [57]:
offers_clean_df = (
    offers_df
    .withColumn(
        "offer_type",
        F.initcap(F.trim(F.lower(F.col("offer_type"))))
    )
    .withColumn(
        "offer_type",
        F.coalesce(
            F.col("offer_type"),
            F.lit("Unknown")
        )
    )
    .withColumn(
        "accepted",
        F.lower(F.trim(F.col("accepted").cast("string")))
    )
    .withColumn(
        "accepted",
        F.when(
            F.col("accepted").isin("true", "yes", "y", "1"),
            F.lit(True)
        )
        .when(
            F.col("accepted").isin("false", "no", "n", "0"),
            F.lit(False)
        )
        .otherwise(F.lit(None))
        .cast("boolean")
    )
    .withColumn(
    "date_offered",
    F.to_date(F.col("date_offered"), "yyyy/MM/dd")
    )
    
)

In [58]:
offers_clean_df.printSchema()

root
 |-- offer_id: integer (nullable = true)
 |-- customer_id: integer (nullable = true)
 |-- offer_type: string (nullable = false)
 |-- accepted: boolean (nullable = true)
 |-- date_offered: date (nullable = true)



In [59]:
offers_clean_df.select(
    "offer_id",
    "customer_id",
    "offer_type",
    "accepted",
    "date_offered"
).show(10, truncate=False)

+--------+-----------+----------------+--------+------------+
|offer_id|customer_id|offer_type      |accepted|date_offered|
+--------+-----------+----------------+--------+------------+
|1       |15784148   |Cashback        |true    |2025-09-03  |
|2       |15687329   |Cashback        |true    |2025-09-04  |
|3       |15738150   |Investment Plan |true    |2025-09-05  |
|4       |15602497   |Cashback        |false   |2025-09-06  |
|5       |15777599   |Insurance Bundle|false   |2025-09-07  |
|6       |15758477   |Cashback        |false   |2025-09-08  |
|7       |15645103   |Insurance Bundle|false   |2025-09-09  |
|8       |15696175   |Personal Loan   |false   |2025-09-10  |
|9       |15800268   |Unknown         |true    |2025-09-11  |
|10      |15658409   |Personal Loan   |false   |2025-09-12  |
+--------+-----------+----------------+--------+------------+
only showing top 10 rows



### Data Quality Validation: verify nulls, duplicates, invalid values, and schemas after cleaning.

In [60]:
def null_report_cleaned(df, name):
    print(f"\n{'=' * 60}")
    print(f"{name} - NULL CHECK")
    print(f"{'=' * 60}")

    null_exprs = [
        F.sum(F.col(c).isNull().cast("int")).alias(c)
        for c in df.columns
    ]

    df.select(null_exprs).show()

In [61]:
null_report_cleaned(customer_clean_df, "Customer")
null_report_cleaned(customer_usage_clean_df, "Customer Usage")
null_report_cleaned(tickets_clean_df, "Tickets")
null_report_cleaned(offers_clean_df, "Offers")


Customer - NULL CHECK


2026-09-04 13:16:38,741 WARN util.package: Truncated the string representation of a plan since it was too large. This behavior can be adjusted by setting 'spark.sql.debug.maxToStringFields'.


+----------+-----------+-------+------------+---------+------+---+------+---------------+-----------+----------------+----------------+------+-------------+---------+-----+------------+-----------+-------+
|row_number|customer_id|surname|credit_score|geography|gender|age|tenure|num_of_products|has_cr_card|is_active_member|estimated_salary|exited|date_of_birth|join_date|email|phone_number|national_id|address|
+----------+-----------+-------+------------+---------+------+---+------+---------------+-----------+----------------+----------------+------+-------------+---------+-----+------------+-----------+-------+
|         0|          0|      0|           0|        0|     0|  0|     0|              0|          0|               0|               0|     0|        10000|    10000|    3|           0|          0|     50|
+----------+-----------+-------+------------+---------+------+---+------+---------------+-----------+----------------+----------------+------+-------------+---------+-----+----

In [62]:
# ============================================================
# DUPLICATE VALIDATION
# ============================================================

print(
    "Customer duplicate customer_id:",
    customer_clean_df.count()
    - customer_clean_df.select("customer_id").distinct().count()
)

print(
    "Customer Usage duplicate usage_id:",
    customer_usage_clean_df.count()
    - customer_usage_clean_df.select("usage_id").distinct().count()
)

print(
    "Tickets duplicate ticket_id:",
    tickets_clean_df.count()
    - tickets_clean_df.select("ticket_id").distinct().count()
)

print(
    "Offers duplicate offer_id:",
    offers_clean_df.count()
    - offers_clean_df.select("offer_id").distinct().count()
)

Customer duplicate customer_id: 0


Customer Usage duplicate usage_id: 0


Tickets duplicate ticket_id: 0


Offers duplicate offer_id: 0


In [63]:
# ============================================================
# INVALID VALUES VALIDATION
# ============================================================

print(
    "Negative resolution times:",
    tickets_clean_df
    .filter(F.col("resolution_time_hrs") < 0)
    .count()
)

Negative resolution times: 93


In [64]:
tickets_clean_df.select(
    "severity"
).distinct().show()

+--------+
|severity|
+--------+
|    High|
|     Low|
|  Medium|
|Critical|
+--------+



In [65]:
offers_clean_df.select(
    "accepted"
).distinct().show()

+--------+
|accepted|
+--------+
|    null|
|    true|
|   false|
+--------+



In [66]:
# ============================================================
# SCHEMA VALIDATION
# ============================================================

print("\n===== CUSTOMER =====")
customer_clean_df.printSchema()

print("\n===== CUSTOMER USAGE =====")
customer_usage_clean_df.printSchema()

print("\n===== TICKETS =====")
tickets_clean_df.printSchema()

print("\n===== OFFERS =====")
offers_clean_df.printSchema()


===== CUSTOMER =====
root
 |-- row_number: integer (nullable = true)
 |-- customer_id: integer (nullable = true)
 |-- surname: string (nullable = true)
 |-- credit_score: integer (nullable = true)
 |-- geography: string (nullable = true)
 |-- gender: string (nullable = true)
 |-- age: integer (nullable = true)
 |-- tenure: integer (nullable = true)
 |-- num_of_products: integer (nullable = true)
 |-- has_cr_card: integer (nullable = true)
 |-- is_active_member: integer (nullable = true)
 |-- estimated_salary: double (nullable = true)
 |-- exited: integer (nullable = true)
 |-- date_of_birth: date (nullable = true)
 |-- join_date: date (nullable = true)
 |-- email: string (nullable = true)
 |-- phone_number: long (nullable = true)
 |-- national_id: long (nullable = true)
 |-- address: string (nullable = true)


===== CUSTOMER USAGE =====
root
 |-- product_type: string (nullable = false)
 |-- usage_id: integer (nullable = true)
 |-- customer_id: integer (nullable = true)
 |-- monthly_ba

In [67]:
# ============================================================
# BUSINESS RULE VALIDATION
# ============================================================

print("Negative resolution times:",
      tickets_clean_df
      .filter(F.col("resolution_time_hrs") < 0)
      .count())

print("Unresolved tickets:",
      tickets_clean_df
      .filter(F.col("is_resolved") == 0)
      .count())

print("Resolved tickets:",
      tickets_clean_df
      .filter(F.col("is_resolved") == 1)
      .count())

Negative resolution times: 93
Unresolved tickets: 390
Resolved tickets: 5669


In [68]:
tickets_clean_df.groupBy("severity").count().show()

+--------+-----+
|severity|count|
+--------+-----+
|    High| 1193|
|     Low| 2143|
|  Medium| 1836|
|Critical|  887|
+--------+-----+



In [69]:
offers_clean_df.groupBy("accepted").count().show()

+--------+-----+
|accepted|count|
+--------+-----+
|    null|    1|
|    true| 2638|
|   false| 5440|
+--------+-----+



## Final Check for Data Quality

In [70]:
print("===== CUSTOMER DATA QUALITY CHECK =====")

print(
    "Duplicate customer_id:",
    customer_clean_df.count()
    - customer_clean_df.select("customer_id").distinct().count()
)

print(
    "Invalid age:",
    customer_clean_df.filter(
        (F.col("age") < 18) | (F.col("age") > 100)
    ).count()
)

print(
    "Invalid credit_score:",
    customer_clean_df.filter(
        (F.col("credit_score") < 0) |
        (F.col("credit_score") > 850)
    ).count()
)

print(
    "Invalid tenure:",
    customer_clean_df.filter(
        F.col("tenure") < 0
    ).count()
)

print("\nGender values:")
customer_clean_df.groupBy("gender").count().show()

print("\nGeography values:")
customer_clean_df.groupBy("geography").count().show()

===== CUSTOMER DATA QUALITY CHECK =====


Duplicate customer_id: 0


Invalid age: 0


Invalid credit_score: 0


Invalid tenure: 0

Gender values:


+------+-----+
|gender|count|
+------+-----+
|Female| 4543|
|  Male| 5457|
+------+-----+


Geography values:


+---------+-----+
|geography|count|
+---------+-----+
|  Germany| 2509|
|   France| 5014|
|    Spain| 2477|
+---------+-----+



In [71]:
print("===== CUSTOMER USAGE DATA QUALITY CHECK =====")

print(
    "Duplicate usage_id:",
    customer_usage_clean_df.count()
    - customer_usage_clean_df.select("usage_id").distinct().count()
)

print(
    "Negative monthly_balance:",
    customer_usage_clean_df.filter(
        F.col("monthly_balance") < 0
    ).count()
)

print(
    "Invalid num_products:",
    customer_usage_clean_df.filter(
        F.col("num_products") <= 0
    ).count()
)

print("\nProduct types:")
customer_usage_clean_df.groupBy("product_type") \
    .count() \
    .orderBy(F.desc("count")) \
    .show(50, truncate=False)

===== CUSTOMER USAGE DATA QUALITY CHECK =====


Duplicate usage_id: 0
Negative monthly_balance: 0
Invalid num_products: 0

Product types:
+------------+-----+
|product_type|count|
+------------+-----+
|Loan        |3022 |
|Checking    |2947 |
|Savings     |2947 |
|Investment  |2926 |
|Credit Card |2922 |
|Unknown     |460  |
+------------+-----+



In [72]:
customer_usage_clean_df = (
    customer_usage_clean_df
    .withColumn(
        "product_type",
        F.when(
            F.trim(F.col("product_type")) == "",
            F.lit("Unknown")
        ).otherwise(F.col("product_type"))
    )
)

In [73]:
print("===== FINAL CUSTOMER USAGE VALIDATION =====")

print(
    "Duplicate usage_id:",
    customer_usage_clean_df.count()
    - customer_usage_clean_df.select("usage_id").distinct().count()
)

print(
    "Negative monthly_balance:",
    customer_usage_clean_df.filter(
        F.col("monthly_balance") < 0
    ).count()
)

print(
    "Invalid num_products:",
    customer_usage_clean_df.filter(
        F.col("num_products") <= 0
    ).count()
)

print(
    "Empty/Unknown product_type:",
    customer_usage_clean_df.filter(
        F.trim(F.col("product_type")) == ""
    ).count()
)

===== FINAL CUSTOMER USAGE VALIDATION =====


Duplicate usage_id: 0
Negative monthly_balance: 0
Invalid num_products: 0
Empty/Unknown product_type: 0


In [74]:
print("===== TICKETS DATA QUALITY CHECK =====")

print(
    "Duplicate ticket_id:",
    tickets_clean_df.count()
    - tickets_clean_df.select("ticket_id").distinct().count()
)

print(
    "Negative resolution_time_hrs:",
    tickets_clean_df.filter(
        F.col("resolution_time_hrs") < 0
    ).count()
)

print(
    "Invalid is_resolved values:",
    tickets_clean_df.filter(
        ~F.col("is_resolved").isin(0, 1)
    ).count()
)

print("\nIssue types:")
tickets_clean_df.groupBy("issue_type") \
    .count() \
    .orderBy(F.desc("count")) \
    .show(50, truncate=False)

print("\nSeverity values:")
tickets_clean_df.groupBy("severity") \
    .count() \
    .orderBy(F.desc("count")) \
    .show()

===== TICKETS DATA QUALITY CHECK =====


Duplicate ticket_id: 0
Negative resolution_time_hrs: 93
Invalid is_resolved values: 0

Issue types:
+---------------+-----+
|issue_type     |count|
+---------------+-----+
|Loan Query     |780  |
|Login Issue    |762  |
|Payment Failure|755  |
|Account Update |754  |
|App Bug        |723  |
|Complaint      |718  |
|Card Problem   |702  |
|Fraud Alert    |690  |
|Unknown        |175  |
+---------------+-----+


Severity values:
+--------+-----+
|severity|count|
+--------+-----+
|     Low| 2143|
|  Medium| 1836|
|    High| 1193|
|Critical|  887|
+--------+-----+



In [75]:
tickets_clean_df = (
    tickets_clean_df
    .withColumn(
        "resolution_time_hrs",
        F.when(
            F.col("resolution_time_hrs") > 0,
            F.col("resolution_time_hrs")
        ).otherwise(F.lit(0))
    )
    .withColumn(
        "is_resolved",
        F.when(
            F.col("resolution_time_hrs") > 0,
            1
        ).otherwise(0)
    )
)

In [76]:
print("===== OFFERS DATA QUALITY CHECK =====")

print(
    "Duplicate offer_id:",
    offers_clean_df.count()
    - offers_clean_df.select("offer_id").distinct().count()
)

print("\nAccepted values:")
offers_clean_df.groupBy("accepted") \
    .count() \
    .orderBy(F.desc("count")) \
    .show()

print("\nOffer types:")
offers_clean_df.groupBy("offer_type") \
    .count() \
    .orderBy(F.desc("count")) \
    .show(50, truncate=False)

===== OFFERS DATA QUALITY CHECK =====


Duplicate offer_id: 0

Accepted values:
+--------+-----+
|accepted|count|
+--------+-----+
|   false| 5440|
|    true| 2638|
|    null|    1|
+--------+-----+


Offer types:
+-------------------+-----+
|offer_type         |count|
+-------------------+-----+
|Personal Loan      |1601 |
|Credit Card Upgrade|1581 |
|Insurance Bundle   |1568 |
|Investment Plan    |1520 |
|Cashback           |1485 |
|Unknown            |324  |
+-------------------+-----+



In [77]:
## FIX EMPTY OFFER TYPES

offers_clean_df = (
    offers_clean_df
    .withColumn(
        "offer_type",
        F.when(
            F.trim(F.col("offer_type")) == "",
            F.lit("Unknown")
        ).otherwise(F.col("offer_type"))
    )
)

In [78]:
print("===== OFFERS DATA QUALITY CHECK =====")

print(
    "Duplicate offer_id:",
    offers_clean_df.count()
    - offers_clean_df.select("offer_id").distinct().count()
)

print("\nAccepted values:")
offers_clean_df.groupBy("accepted") \
    .count() \
    .orderBy(F.desc("count")) \
    .show()

print("\nOffer types:")
offers_clean_df.groupBy("offer_type") \
    .count() \
    .orderBy(F.desc("count")) \
    .show(50, truncate=False)

===== OFFERS DATA QUALITY CHECK =====


Duplicate offer_id: 0

Accepted values:
+--------+-----+
|accepted|count|
+--------+-----+
|   false| 5440|
|    true| 2638|
|    null|    1|
+--------+-----+


Offer types:
+-------------------+-----+
|offer_type         |count|
+-------------------+-----+
|Personal Loan      |1601 |
|Credit Card Upgrade|1581 |
|Insurance Bundle   |1568 |
|Investment Plan    |1520 |
|Cashback           |1485 |
|Unknown            |324  |
+-------------------+-----+



# Store the Silver layer Data

## Store them as HDFS Files

In [79]:
customer_clean_df.write \
    .format(OUTPUT_FORMAT) \
    .mode("overwrite") \
    .save(CUSTOMER_SILVER_PATH)

customer_usage_clean_df.write \
    .format(OUTPUT_FORMAT) \
    .mode("overwrite") \
    .save(CUSTOMER_USAGE_SILVER_PATH)

tickets_clean_df.write \
    .format(OUTPUT_FORMAT) \
    .mode("overwrite") \
    .save(TICKETS_SILVER_PATH)

offers_clean_df.write \
    .format(OUTPUT_FORMAT) \
    .mode("overwrite") \
    .save(OFFERS_SILVER_PATH)

print("Silver layer written successfully.")

Silver layer written successfully.


## Make Hive External tables points on my silver data

In [80]:
spark.sql(f"""
    CREATE TABLE IF NOT EXISTS {HIVE_DB}.customer_silver
    USING PARQUET
    LOCATION '{CUSTOMER_SILVER_PATH}'
""")

spark.sql(f"""
    CREATE TABLE IF NOT EXISTS {HIVE_DB}.customer_usage_silver
    USING PARQUET
    LOCATION '{CUSTOMER_USAGE_SILVER_PATH}'
""")

spark.sql(f"""
    CREATE TABLE IF NOT EXISTS {HIVE_DB}.tickets_silver
    USING PARQUET
    LOCATION '{TICKETS_SILVER_PATH}'
""")

spark.sql(f"""
    CREATE TABLE IF NOT EXISTS {HIVE_DB}.offers_silver
    USING PARQUET
    LOCATION '{OFFERS_SILVER_PATH}'
""")

DataFrame[]

### Validate

In [81]:
print("===== HIVE TABLES =====")
spark.sql("SHOW TABLES").show()

print("===== CUSTOMER TABLE =====")
spark.sql("DESCRIBE customer_silver").show()

print("===== CUSTOMER SAMPLE =====")
spark.sql("""
    SELECT *
    FROM customer_silver
    LIMIT 10
""").show(truncate=False)

print("===== CUSTOMER COUNT =====")
spark.sql("""
    SELECT COUNT(*) AS total_customers
    FROM customer_silver
""").show()

print("Silver tables registered successfully.")

===== HIVE TABLES =====
+-----------------+--------------------+-----------+
|         database|           tableName|isTemporary|
+-----------------+--------------------+-----------+
|customer_churn_db|     customer_silver|      false|
|customer_churn_db|customer_usage_si...|      false|
|customer_churn_db|       offers_silver|      false|
|customer_churn_db|      tickets_silver|      false|
+-----------------+--------------------+-----------+

===== CUSTOMER TABLE =====
+----------------+---------+-------+
|        col_name|data_type|comment|
+----------------+---------+-------+
|      row_number|      int|   null|
|     customer_id|      int|   null|
|         surname|   string|   null|
|    credit_score|      int|   null|
|       geography|   string|   null|
|          gender|   string|   null|
|             age|      int|   null|
|          tenure|      int|   null|
| num_of_products|      int|   null|
|     has_cr_card|      int|   null|
|is_active_member|      int|   null|
|estim

+---------------+
|total_customers|
+---------------+
|          10000|
+---------------+

Silver tables registered successfully.


# Read from the Silver Layer

In [82]:
customer_silver_df = spark.table(f"{HIVE_DB}.customer_silver")
usage_silver_df = spark.table(f"{HIVE_DB}.customer_usage_silver")
tickets_silver_df = spark.table(f"{HIVE_DB}.tickets_silver")
offers_silver_df = spark.table(f"{HIVE_DB}.offers_silver")

# Calculate Business requirement

## Prepare Activity Month

In [83]:
usage_monthly_base = (
    usage_silver_df
    .withColumn(
        "activity_month",
        F.trunc(F.col("date"), "month")
    )
)

tickets_monthly_base = (
    tickets_silver_df
    .withColumn(
        "activity_month",
        F.trunc(F.col("date"), "month")
    )
)

offers_monthly_base = (
    offers_silver_df
    .withColumn(
        "activity_month",
        F.trunc(F.col("date_offered"), "month")
    )
)

In [84]:
usage_monthly_base.show()

+------------+--------+-----------+---------------+------------+----------+--------------+
|product_type|usage_id|customer_id|monthly_balance|num_products|      date|activity_month|
+------------+--------+-----------+---------------+------------+----------+--------------+
|    Checking|       1|   15665177|         445.51|           2|2026-04-08|    2026-04-01|
|     Savings|       2|   15756472|       36053.68|           1|2025-12-19|    2025-12-01|
|     Savings|       3|   15589881|         320.82|           2|2026-04-05|    2026-04-01|
|     Savings|       4|   15584091|       54886.26|           2|2026-08-27|    2026-08-01|
|    Checking|       5|   15653306|       60361.31|           1|2026-03-12|    2026-03-01|
|    Checking|       6|   15762392|      103960.52|           1|2026-03-08|    2026-03-01|
| Credit Card|       7|   15725639|      130175.17|           1|2026-03-27|    2026-03-01|
|    Checking|       8|   15725002|         603.99|           2|2026-01-24|    2026-01-01|

## The relation between usage and customer 
- Make a window function to partition by customer id and activity month with ordering by data
- calculate the latest usage record for each customer-month
- count the number of distinct product
- Join the latest balance to the product count

In [85]:
# A window for partitioning
usage_window = (
    Window
    .partitionBy("customer_id", "activity_month")
    .orderBy(F.col("date").desc())
)
usage_latest_df = (
    usage_monthly_base
    .withColumn("row_num", F.row_number().over(usage_window))
    .filter(F.col("row_num") == 1)
    .drop("row_num")
)

In [86]:
usage_agg_df = (
    usage_monthly_base
    .groupBy("customer_id", "activity_month")
    .agg(
        F.countDistinct("product_type").alias("active_products_count")
    )
)

In [87]:
usage_agg_df = (
    usage_agg_df
    .join(
        usage_latest_df.select(
            "customer_id",
            "activity_month",
            F.col("monthly_balance").alias("ending_monthly_balance")
        ),
        on=["customer_id", "activity_month"],
        how="left"
    )
)

In [88]:
usage_agg_df.show()

+-----------+--------------+---------------------+----------------------+
|customer_id|activity_month|active_products_count|ending_monthly_balance|
+-----------+--------------+---------------------+----------------------+
|   15797227|    2026-04-01|                    1|                912.09|
|   15734634|    2026-02-01|                    1|             124574.22|
|   15693543|    2026-01-01|                    2|                632.04|
|   15572291|    2026-06-01|                    1|             190319.53|
|   15734610|    2025-10-01|                    1|              40737.86|
|   15706217|    2026-05-01|                    1|              59101.65|
|   15640418|    2025-10-01|                    1|             142417.08|
|   15594594|    2026-07-01|                    1|              88098.72|
|   15755678|    2025-12-01|                    1|                638.77|
|   15745399|    2026-07-01|                    1|                911.36|
|   15741643|    2026-05-01|          

## Tickets monthly aggregation
Grouping by customer id and activity month, I calculate the:
- Number of opened tickets using count(*)
- numebr of critical tickets
- total resolution time

In [89]:
tickets_agg_df = (
    tickets_monthly_base
    .groupBy("customer_id", "activity_month")
    .agg(
        F.count("*").alias("tickets_opened_count"),

        F.sum(
            F.when(
                F.lower(F.col("severity")) == "critical",
                1
            ).otherwise(0)
        ).alias("critical_tickets_count"),

        F.sum("resolution_time_hrs")
            .alias("total_resolution_time_hrs")
    )
)

In [90]:
tickets_agg_df.show()

+-----------+--------------+--------------------+----------------------+-------------------------+
|customer_id|activity_month|tickets_opened_count|critical_tickets_count|total_resolution_time_hrs|
+-----------+--------------+--------------------+----------------------+-------------------------+
|   15807312|    2026-08-01|                   1|                     0|                      0.0|
|   15621685|    2026-06-01|                   1|                     0|                    129.4|
|   15568120|    2026-02-01|                   1|                     1|                      0.8|
|   15682778|    2025-10-01|                   1|                     0|                     48.5|
|   15797227|    2026-04-01|                   1|                     0|                     90.4|
|   15791316|    2026-06-01|                   1|                     0|                     19.3|
|   15577683|    2026-05-01|                   1|                     0|                     24.8|
|   156402

## Offers monthly aggregation
BY grouping by customer id and activity month, I calculate the:
- offers received count
- offers accepted count

In [91]:
offers_agg_df = (
    offers_monthly_base
    .groupBy("customer_id", "activity_month")
    .agg(
        F.count("*").alias("offers_received_count"),

        F.sum(
            F.when(F.col("accepted") == True, 1)
             .otherwise(0)
        ).alias("offers_accepted_count")
    )
)

In [92]:
offers_agg_df.show()

+-----------+--------------+---------------------+---------------------+
|customer_id|activity_month|offers_received_count|offers_accepted_count|
+-----------+--------------+---------------------+---------------------+
|   15814664|    2026-06-01|                    3|                    1|
|   15747014|    2025-09-01|                    1|                    1|
|   15761286|    2025-09-01|                    1|                    0|
|   15660636|    2025-10-01|                    1|                    0|
|   15755678|    2025-12-01|                    1|                    0|
|   15797227|    2026-04-01|                    1|                    1|
|   15803456|    2025-10-01|                    1|                    0|
|   15679801|    2025-11-01|                    1|                    1|
|   15771997|    2026-08-01|                    1|                    1|
|   15724127|    2026-06-01|                    1|                    0|
|   15654390|    2025-09-01|                    1| 

## Monthly activity
To build monthly activity, you should know that it's not proper to have the three events happening in the same month, so we need to get the active months first then get the events that happened in this month
### Get the active months

In [94]:
activity_spine_df = (
    usage_agg_df
    .select("customer_id", "activity_month")
    .union(
        tickets_agg_df.select("customer_id", "activity_month")
    )
    .union(
        offers_agg_df.select("customer_id", "activity_month")
    )
    .distinct()
)

### know the specific events that happened in the active months

In [95]:
monthly_activity_df = activity_spine_df \
    .join(usage_agg_df, ["customer_id", "activity_month"], "left") \
    .join(tickets_agg_df, ["customer_id", "activity_month"], "left") \
    .join(offers_agg_df, ["customer_id", "activity_month"], "left")

### Fix the nulls
As this join will lead us to have null values in the unhappened events so we need to replace them except the balance because the balance can carry two meanings:
    1. Having the balance 0$
    2. Not having balance record for this month
So we need to keep it null as it is.

In [96]:
monthly_activity_df = (
    monthly_activity_df
    .fillna({
        "offers_received_count": 0,
        "offers_accepted_count": 0,
        "tickets_opened_count": 0,
        "critical_tickets_count": 0,
        "total_resolution_time_hrs": 0,
        "active_products_count": 0
    })
)

In [97]:
monthly_activity_df.show(20, truncate=False)
monthly_activity_df.printSchema()

+-----------+--------------+---------------------+----------------------+--------------------+----------------------+-------------------------+---------------------+---------------------+
|customer_id|activity_month|active_products_count|ending_monthly_balance|tickets_opened_count|critical_tickets_count|total_resolution_time_hrs|offers_received_count|offers_accepted_count|
+-----------+--------------+---------------------+----------------------+--------------------+----------------------+-------------------------+---------------------+---------------------+
|15566269   |2025-09-01    |1                    |290.36                |0                   |0                     |0.0                      |0                    |0                    |
|15568120   |2026-02-01    |0                    |null                  |1                   |1                     |0.8                      |0                    |0                    |
|15571284   |2026-07-01    |2                    |35254.14  

# Build The dimension tables

## Data Dimension

### Get the active dates
I can get them from activity_spine_df but this df is the latest one and besides I will use it in my facts

In [98]:
activity_dates_df = (
    monthly_activity_df
    .select("activity_month")
    .distinct()
)

### Create Date dimension

In [ ]:
dim_date_df = (
    activity_dates_df
    .withColumnRenamed("activity_month", "full_date")

    .withColumn(
        "date_key",
        F.date_format("full_date", "yyyyMMdd").cast("int")
    )

    .withColumn(
        "calendar_month",
        F.month("full_date")
    )

    .withColumn(
        "month_name",
        F.date_format("full_date", "MMMM")
    )

    .withColumn(
        "calendar_quarter",
        F.quarter("full_date")
    )

    .withColumn(
        "calendar_year",
        F.year("full_date")
    )

    .select(
        "date_key",
        "full_date",
        "calendar_month",
        "month_name",
        "calendar_quarter",
        "calendar_year"
    )
)

In [100]:
dim_date_df.show()
dim_date_df.printSchema()

+--------+----------+--------------+----------+----------------+-------------+
|date_key| full_date|calendar_month|month_name|calendar_quarter|calendar_year|
+--------+----------+--------------+----------+----------------+-------------+
|20260201|2026-02-01|             2|  February|               1|         2026|
|20260801|2026-08-01|             8|    August|               3|         2026|
|20260501|2026-05-01|             5|       May|               2|         2026|
|20260701|2026-07-01|             7|      July|               3|         2026|
|20260901|2026-09-01|             9| September|               3|         2026|
|20251001|2025-10-01|            10|   October|               4|         2025|
|20251201|2025-12-01|            12|  December|               4|         2025|
|20260401|2026-04-01|             4|     April|               2|         2026|
|20250901|2025-09-01|             9| September|               3|         2025|
|20260301|2026-03-01|             3|     March|     

## Create Customer_dim

This happens by:

1. Select the target attributes from the customer silver table.

2. Calculate the surrogate key using the window function `row_number()`.

3. Calculate `is_churned` by converting `exited` into boolean values.

4. Use the Slowly Changing Dimension (SCD) Type 2:

   1. Calculate the start date as the current date.
   2. Calculate the end date using a big default value `9999-12-31` to show that this record is still active.
   3. Use `is_current` to check if the record is the current active version.

5. Drop `exited` because there is no need for it anymore after creating `is_churned`.

6. Select the final attributes for the `dim_customer` table.

In [103]:
customer_key_window = Window.orderBy("customer_id")

dim_customer_df = (
    customer_silver_df

    .select(
        "customer_id",
        "surname",
        "gender",
        "date_of_birth",
        "age",
        "geography",
        "credit_score",
        "tenure",
        "join_date",
        "estimated_salary",
        "has_cr_card",
        "is_active_member",
        "exited"
    )

    .withColumn(
        "cust_key",
        F.row_number().over(customer_key_window)
    )

    .withColumn(
        "is_churned",
        F.col("exited").cast("boolean")
    )

    .withColumn(
        "dw_start_date",
        F.current_date()
    )

    .withColumn(
        "dw_end_date",
        F.to_date(F.lit("9999-12-31"))
    )

    .withColumn(
        "is_current",
        F.lit(True)
    )

    .drop("exited")

    .select(
        "cust_key",
        "customer_id",
        "surname",
        "gender",
        "date_of_birth",
        "age",
        "geography",
        "credit_score",
        "tenure",
        "join_date",
        "estimated_salary",
        "has_cr_card",
        "is_active_member",
        "is_churned",
        "dw_start_date",
        "dw_end_date",
        "is_current"
    )
)

In [104]:
dim_customer_df.show(10, truncate=False)
dim_customer_df.printSchema()

2026-09-04 14:18:10,016 WARN window.WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.


+--------+-----------+--------+------+-------------+---+---------+------------+------+---------+----------------+-----------+----------------+----------+-------------+-----------+----------+
|cust_key|customer_id|surname |gender|date_of_birth|age|geography|credit_score|tenure|join_date|estimated_salary|has_cr_card|is_active_member|is_churned|dw_start_date|dw_end_date|is_current|
+--------+-----------+--------+------+-------------+---+---------+------------+------+---------+----------------+-----------+----------------+----------+-------------+-----------+----------+
|1       |15565701   |Ferri   |Female|null         |39 |Spain    |698         |9     |null     |90212.38        |0          |0               |false     |2026-09-04   |9999-12-31 |true      |
|2       |15565706   |Akobundu|Male  |null         |35 |Spain    |612         |1     |null     |83256.26        |1          |1               |true      |2026-09-04   |9999-12-31 |true      |
|3       |15565714   |Cattaneo|Male  |null   

## Fact customer activity per month creation
### 1. join the customer surrogent key to build the relation between the customer_dim and the fact
Why left join?
    Because I need to get each active month even if there is no customer associated with that month

In [105]:
fact_customer_monthly_activity_df  = (
    monthly_activity_df
    .join(
        dim_customer_df.select(
            "cust_key",
            "customer_id"
        ),
        on="customer_id",
        how="left"
    )
)

### 2. Join the date key to build the relation between the date_dim and the fact

In [111]:
fact_customer_monthly_activity_df = (
    fact_customer_monthly_activity_df
    .join(
        dim_date_df.select(
            "date_key",
            F.col("full_date").alias("activity_month")
        ),
        on="activity_month",
        how="left"
    )
)

### Build the fact table

In [113]:
fact_customer_monthly_activity_df = (
    fact_customer_monthly_activity_df
    .withColumn(
        "activity_key",
        F.monotonically_increasing_id()
    )

    .withColumn(
        "load_timestamp",
        F.current_timestamp()
    )

    .select(
        "activity_key",
        "cust_key",
        "date_key",
        "offers_received_count",
        "offers_accepted_count",
        "tickets_opened_count",
        "critical_tickets_count",
        "total_resolution_time_hrs",
        "ending_monthly_balance",
        "active_products_count",
        "load_timestamp"
    )
)

### Verification

In [112]:
fact_customer_monthly_activity_df.printSchema()

root
 |-- activity_month: date (nullable = true)
 |-- customer_id: integer (nullable = true)
 |-- active_products_count: long (nullable = false)
 |-- ending_monthly_balance: double (nullable = true)
 |-- tickets_opened_count: long (nullable = false)
 |-- critical_tickets_count: long (nullable = false)
 |-- total_resolution_time_hrs: double (nullable = false)
 |-- offers_received_count: long (nullable = false)
 |-- offers_accepted_count: long (nullable = false)
 |-- cust_key: integer (nullable = true)
 |-- date_key: integer (nullable = true)



In [115]:
fact_customer_monthly_activity_df.show(20, truncate=False)

2026-09-04 14:36:29,505 WARN window.WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.


+------------+--------+--------+---------------------+---------------------+--------------------+----------------------+-------------------------+----------------------+---------------------+-----------------------+
|activity_key|cust_key|date_key|offers_received_count|offers_accepted_count|tickets_opened_count|critical_tickets_count|total_resolution_time_hrs|ending_monthly_balance|active_products_count|load_timestamp         |
+------------+--------+--------+---------------------+---------------------+--------------------+----------------------+-------------------------+----------------------+---------------------+-----------------------+
|34359738368 |89      |20260201|0                    |0                    |1                   |1                     |0.8                      |null                  |0                    |2026-09-04 14:36:28.467|
|34359738369 |1799    |20260201|0                    |0                    |1                   |0                     |16.3            

The two should be equal

In [117]:
print(
    "Fact rows:",
    fact_customer_monthly_activity_df.count()
)

print(
    "Distinct customer-month rows:",
    monthly_activity_df.select(
        "customer_id",
        "activity_month"
    ).distinct().count()
)

Fact rows: 25848


Distinct customer-month rows: 25848


In [118]:
monthly_activity_df.groupBy(
    "customer_id",
    "activity_month"
).count().filter(
    F.col("count") > 1
).show()

+-----------+--------------+-----+
|customer_id|activity_month|count|
+-----------+--------------+-----+
+-----------+--------------+-----+



# write the Gold layer to HDFS

In [128]:
dim_customer_df.write \
    .format("parquet") \
    .mode("overwrite") \
    .save(DIM_CUSTOMER_PATH)

dim_date_df.write \
    .format("parquet") \
    .mode("overwrite") \
    .save(DIM_DATE_PATH)

fact_customer_monthly_activity_df.write \
    .format("parquet") \
    .mode("overwrite") \
    .save(FACT_ACTIVITY_PATH)

print("Gold Parquet data written successfully.")

2026-09-04 14:55:33,625 WARN window.WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
2026-09-04 14:55:49,622 WARN window.WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.


Gold Parquet data written successfully.


# Build Gold tables in Hive

In [133]:
spark.sql(f"""
    CREATE TABLE IF NOT EXISTS {HIVE_DB}.dim_customer
    USING PARQUET
    LOCATION '{DIM_CUSTOMER_PATH}'
""")
spark.sql(f"""
    CREATE TABLE IF NOT EXISTS {HIVE_DB}.dim_date
    USING PARQUET
    LOCATION '{DIM_DATE_PATH}'
""")
spark.sql(f"""
    CREATE TABLE IF NOT EXISTS {HIVE_DB}.fact_customer_monthly_activity_df
    USING PARQUET
    LOCATION '{FACT_ACTIVITY_PATH}'
""")

DataFrame[]

In [134]:
spark.sql("SHOW TABLES").show()

+-----------------+--------------------+-----------+
|         database|           tableName|isTemporary|
+-----------------+--------------------+-----------+
|customer_churn_db|     customer_silver|      false|
|customer_churn_db|customer_usage_si...|      false|
|customer_churn_db|        dim_customer|      false|
|customer_churn_db|            dim_date|      false|
|customer_churn_db|fact_customer_mon...|      false|
|customer_churn_db|       offers_silver|      false|
|customer_churn_db|      tickets_silver|      false|
+-----------------+--------------------+-----------+



# Validate the warehouse
Check the counts

In [135]:
spark.sql("""
    SELECT COUNT(*) AS customer_count
    FROM dim_customer
""").show()

spark.sql("""
    SELECT COUNT(*) AS date_count
    FROM dim_date
""").show()

+--------------+
|customer_count|
+--------------+
|         10000|
+--------------+

+----------+
|date_count|
+----------+
|        13|
+----------+



In [136]:
spark.sql("""
    SELECT COUNT(*) AS fact_count
    FROM fact_customer_monthly_activity_df
""").show()

+----------+
|fact_count|
+----------+
|     25848|
+----------+



Check that the grain is only one row

In [137]:
spark.sql("""
    SELECT
        cust_key,
        date_key,
        COUNT(*) AS record_count
    FROM fact_customer_monthly_activity_df
    GROUP BY cust_key, date_key
    HAVING COUNT(*) > 1
""").show()

+--------+--------+------------+
|cust_key|date_key|record_count|
+--------+--------+------------+
+--------+--------+------------+

